In [1]:
import warnings

from catboost import CatBoostClassifier
import pandas as pd
import numpy as np
import json, optuna, os

warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold

def eval_classification(y_true, y_pred, name=""):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")
    print(f"{name} Accuracy: {acc:.4f} | F1-macro: {f1_macro:.4f} | F1-weighted: {f1_weighted:.4f}")
    return acc, f1_macro, f1_weighted

def save_preds_csv(X_test, y_true, y_pred, filename):
    out = X_test.copy()
    out["y_true"] = y_true
    out["y_pred"] = y_pred
    out.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"Saved: {filename}")

def derive_groups_from_train(train_df, k_baseline_dict, min_group_size=2):
    """
    Returns: type_to_group dict, groups dict
    Uses only train_df to avoid test leakage.
    Phân loại các 'type' thành near/mid/far dựa trên trung bình độ lệch tuyệt đối (abs_diff)
    """
    df_temp = train_df.copy()
    df_temp["k_eff_expected"] = df_temp["load"].map(k_baseline_dict)
    if df_temp["k_eff_expected"].isnull().any():
        known_loads = np.array(list(k_baseline_dict.keys()))
        for idx, row in df_temp[df_temp["k_eff_expected"].isnull()].iterrows():
            closest_load = known_loads[np.abs(known_loads - row["load"]).argmin()]
            df_temp.loc[idx, "k_eff_expected"] = k_baseline_dict[closest_load]     
    df_temp["abs_diff"] = (df_temp["efficiency"] - df_temp["k_eff_expected"]).abs()
    type_stats = (
        df_temp.groupby("type")["abs_diff"]
        .agg(n="count", mean="mean")
        .sort_values("mean")
        .reset_index()
    )
    types = type_stats["type"].tolist()
    x = type_stats["mean"].to_numpy()
    n = len(x)
    # Prefix sums for O(1) segment SSE
    S1 = np.concatenate([[0.0], np.cumsum(x)])
    S2 = np.concatenate([[0.0], np.cumsum(x * x)])
    def seg_sse(i, j):
        m = j - i + 1
        sum1 = S1[j + 1] - S1[i]
        sum2 = S2[j + 1] - S2[i]
        mu = sum1 / m
        return sum2 - 2 * mu * sum1 + m * mu * mu

    K = 3
    INF = 1e18
    dp = np.full((K + 1, n + 1), INF)
    cut = np.full((K + 1, n + 1), -1, dtype=int)

    dp[0, 0] = 0.0
    for k in range(1, K + 1):
        for t in range(1, n + 1):
            p_min = (k - 1) * min_group_size
            p_max = t - min_group_size
            if p_max < p_min:
                continue
            best_cost, best_p = INF, -1
            for p in range(p_min, p_max + 1):
                cost = dp[k - 1, p] + seg_sse(p, t - 1)
                if cost < best_cost:
                    best_cost, best_p = cost, p
            dp[k, t] = best_cost
            cut[k, t] = best_p

    if np.isinf(dp[K, n]):
        raise RuntimeError("No valid split found. Try min_group_size=1 or check types count.")
    bounds = []
    t = n
    for k in range(K, 0, -1):
        p = cut[k, t]
        bounds.append((p, t))
        t = p
    bounds.reverse()
    group_names = ["near", "mid", "far"] 
    groups = {}
    for name, (a, b) in zip(group_names, bounds):
        groups[name] = types[a:b]

    type_to_group = {t: g for g, ts in groups.items() for t in ts}
    return type_to_group, groups, type_stats

def catboost_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 600, 1400, step=200),
        "depth": trial.suggest_int("depth", 4, 8), 
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.5, 3.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "auto_class_weights": "Balanced",
        "loss_function": "MultiClass",
        "random_seed": 42,
        "verbose": False,
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    f1s = []
    X = X_train_r
    y = y_train_r
    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        model = CatBoostClassifier(**params)
        pipe_cv = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", model),
        ])
        pipe_cv.fit(X_tr, y_tr)
        pred = pipe_cv.predict(X_va).ravel()
        f1s.append(f1_score(y_va, pred, average="macro"))

    return float(np.mean(f1s))


# ----- Load your saved split files -----
train_random = pd.read_csv("C:\\Users\\NFU_Power_Lab\\Desktop\\DAT_ResearchArea\\BestAwardAI2026\\GenerativeModel\\CTGANGenData\\wpt_train_random_aug_ctgan.csv")
test_random  = pd.read_csv("C:\\Users\\NFU_Power_Lab\\Desktop\\DAT_ResearchArea\\BestAwardAI2026\\random-split\\wpt_test_random.csv")
catboost_json_path = r"C:\Users\NFU_Power_Lab\Desktop\DAT_ResearchArea\BestAwardAI2026\BaseModel\CatBoost\CatBoostCTGan\catboost-ctgan.json"

# Sanity check expected columns
expected_cols = {"type", "load", "efficiency"}
for name, d in [("train_random", train_random), ("test_random", test_random)]:
    missing = expected_cols - set(d.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

# Make sure numeric columns are numeric
for d in [train_random, test_random]:
    d["load"] = pd.to_numeric(d["load"], errors="coerce")
    d["efficiency"] = pd.to_numeric(d["efficiency"], errors="coerce")
    d["type"] = d["type"].astype(str)
    d.dropna(subset=["type", "load", "efficiency"], inplace=True)

# Features / target
FEATURES = ["load", "efficiency"]
TARGET = "type"

# =========================
def add_advanced_features(df, k_baseline_dict, type_to_group_dict):
    df = df.copy()
    df["k_eff_expected"] = df["load"].map(k_baseline_dict)
    if df["k_eff_expected"].isnull().any():
        known_loads = np.array(list(k_baseline_dict.keys()))
        for idx, row in df[df["k_eff_expected"].isnull()].iterrows():
            closest_load = known_loads[np.abs(known_loads - row["load"]).argmin()]
            df.loc[idx, "k_eff_expected"] = k_baseline_dict[closest_load]  
    # 2. Độ lệch (Difference) và Tỷ lệ (Ratio)
    df["eff_diff"] = df["efficiency"] - df["k_eff_expected"]
    df["eff_ratio"] = df["efficiency"] / df["k_eff_expected"]
    # 3. Ước tính tổn hao công suất (Power Loss Proxy)
    # P_loss = Load * (1 / (Efficiency / 100) - 1)
    df["p_loss"] = df["load"] * (1 / (df["efficiency"] / 100.0) - 1)
    # Tổn hao công suất của Class K
    df["k_p_loss"] = df["load"] * (1 / (df["k_eff_expected"] / 100.0) - 1)
    # Độ lệch suy hao do vật thể lạ gây ra
    df["p_loss_diff"] = df["p_loss"] - df["k_p_loss"]

    df["dist_zone_name"] = df["type"].map(type_to_group_dict).fillna("far")
    # Mã hóa thành số
    zone_mapping = {"near": 1, "mid": 2, "far": 3}
    df["dist_zone"] = df["dist_zone_name"].map(zone_mapping)
    # 4. Phân vùng Dải tải (Load Zone)
    # Giả sử dải tải từ 0.5 đến 3.0, chia làm 3 vùng: Nhẹ (1), Vừa (2), Nặng (3)
    df['load_zone'] = pd.cut(df['load'], bins=3, labels=[1, 2, 3]).astype(float)
    return df

k_baseline_train_r = train_random[train_random["type"] == "K"].groupby("load")["efficiency"].mean().to_dict()
type_to_group, groups_dict, stats = derive_groups_from_train(train_random, k_baseline_train_r)
print("Results of grouping the Types:", groups_dict)

train_random_fe = add_advanced_features(train_random, k_baseline_train_r, type_to_group)
test_random_fe = add_advanced_features(test_random, k_baseline_train_r, type_to_group)

FEATURES_NEW = [
    "load", "efficiency", 
    "eff_diff", "eff_ratio", 
    "p_loss_diff", "load_zone", "dist_zone"
]
# =========================
X_train_r = train_random_fe[FEATURES_NEW]
y_train_r = train_random_fe[TARGET]
X_test_r  = test_random_fe[FEATURES_NEW]
y_test_r  = test_random_fe[TARGET]

# ========================
# Choose a baseline classifier
# =========================
OPTUNA_TRIALS = 150
study = optuna.create_study(direction="maximize")
study.optimize(catboost_objective, n_trials=OPTUNA_TRIALS)

print("\n=== Optuna done for CatBoost ===")
print("Best macro-F1 (CV):", study.best_value)
print("Best params:", study.best_params)

best_params = study.best_params
os.makedirs(os.path.dirname(catboost_json_path), exist_ok=True)

with open(catboost_json_path, 'w', encoding='utf-8') as f:
    json.dump(best_params, f, indent=4)
print(f"Saving optuna parameters: {catboost_json_path}")

clf = CatBoostClassifier(
    iterations=best_params.get("iterations", 1000),
    depth=best_params.get("depth", 6),
    learning_rate=best_params.get("learning_rate", 0.05),
    l2_leaf_reg=best_params.get("l2_leaf_reg", 3.0),
    random_strength=best_params.get("random_strength", 1.0),
    bagging_temperature=best_params.get("bagging_temperature", 0.0),
    auto_class_weights='Balanced',
    loss_function='MultiClass',
    random_seed=42,
    verbose=False
)

# Pipeline: scale numeric features (safe for most models)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", clf),
])

# =========================
# Train/Eval on Random Split
# =========================
pipe.fit(X_train_r, y_train_r)
pred_r = pipe.predict(X_test_r).ravel()

print("=== Baseline Classification on Random Split ===")
eval_classification(y_test_r, pred_r, name="RandomSplit")

print("\nClassification report (RandomSplit):")
print(classification_report(y_test_r, pred_r, digits=4))

cm_r = confusion_matrix(y_test_r, pred_r, labels=sorted(y_train_r.unique()))
cm_r_df = pd.DataFrame(cm_r, index=sorted(y_train_r.unique()), columns=sorted(y_train_r.unique()))
cm_r_df.to_csv("baseline_cm_random_ctgan.csv", encoding="utf-8-sig")
print("Saved: baseline_cm_random_ctgan.csv")

save_preds_csv(X_test_r, y_test_r.values, pred_r, "baseline_preds_random_cls_ctgan.csv")

[I 2026-03-12 18:13:59,194] A new study created in memory with name: no-name-60be201c-6c2d-4175-a9f2-9f58f1813641


Results of grouping the Types: {'near': ['K', 'H', 'J', 'G'], 'mid': ['E', 'D', 'I', 'B', 'F'], 'far': ['C', 'A']}


[I 2026-03-12 18:14:36,019] Trial 0 finished with value: 0.6956619115011736 and parameters: {'iterations': 1200, 'depth': 6, 'learning_rate': 0.037730246709537986, 'l2_leaf_reg': 13.796453619613299, 'random_strength': 0.989240257144728, 'bagging_temperature': 0.756049929358476}. Best is trial 0 with value: 0.6956619115011736.
[I 2026-03-12 18:14:59,069] Trial 1 finished with value: 0.6868756009558842 and parameters: {'iterations': 1400, 'depth': 5, 'learning_rate': 0.12547493577934823, 'l2_leaf_reg': 13.604135095570815, 'random_strength': 0.6184359160227146, 'bagging_temperature': 0.26419536120354137}. Best is trial 0 with value: 0.6956619115011736.
[I 2026-03-12 18:15:50,517] Trial 2 finished with value: 0.6793159715911776 and parameters: {'iterations': 1000, 'depth': 7, 'learning_rate': 0.056949006670221285, 'l2_leaf_reg': 2.029392407503278, 'random_strength': 1.9900451621570552, 'bagging_temperature': 0.7071472345705668}. Best is trial 0 with value: 0.6956619115011736.
[I 2026-03-12


=== Optuna done for CatBoost ===
Best macro-F1 (CV): 0.7088978273379145
Best params: {'iterations': 1400, 'depth': 4, 'learning_rate': 0.0304176202050611, 'l2_leaf_reg': 12.919272325122199, 'random_strength': 1.8711511365896056, 'bagging_temperature': 0.00016969203343876818}
Saving optuna parameters: C:\Users\NFU_Power_Lab\Desktop\DAT_ResearchArea\BestAwardAI2026\BaseModel\CatBoost\CatBoostCTGan\catboost-ctgan.json
=== Baseline Classification on Random Split ===
RandomSplit Accuracy: 0.7877 | F1-macro: 0.7644 | F1-weighted: 0.7851

Classification report (RandomSplit):
              precision    recall  f1-score   support

           A     1.0000    1.0000    1.0000         4
           B     1.0000    0.8000    0.8889        15
           C     1.0000    1.0000    1.0000         2
           D     0.6000    1.0000    0.7500         9
           E     1.0000    0.6154    0.7619        13
           F     0.0000    0.0000    0.0000         0
           G     1.0000    1.0000    1.0000  